In [55]:
# Merge the train datasets into one file for easier loading
import os
import pandas as pd
from glob import glob
from tqdm import tqdm



def merge_datasets(data_dir):
    # Get a list of all CSV files in the directory
    csv_files = glob(os.path.join(data_dir, "*.csv"))

    # Initialize an empty list to hold the DataFrames
    dfs = []

    # Loop through the CSV files and read them into DataFrames
    for csv_file in tqdm(csv_files, desc="Loading data"):
        df = pd.read_csv(csv_file)
        dfs.append(df)
    
    # merge all the DataFrames into one by Latitude and Longitude
    # so if df1 has Latitude and Longitude columns, and df2 has Latitude and Longitude columns, then we merge them on those columns
    merged_df = dfs[0]
    for df in dfs[1:]:
        merged_df = pd.merge(merged_df, df, on=["Latitude", "Longitude", "Sample Date"], how="outer")
    
    return merged_df

# training datasets are in the "data/training_data" directory
train_data_dir = "../data/training_data"

trained_df = merge_datasets(train_data_dir)

# Save the merged DataFrame to a new CSV file
output_file = "../data/merged_training_data.csv"
trained_df.to_csv(output_file, index=False)
print(f"Merged training dataset saved to {output_file}")

# testing datasets are in the "data/testing_data" directory
test_data_dir = "../data/testing_data"

tested_df = merge_datasets(test_data_dir)

# Save the merged DataFrame to a new CSV file
output_file = "../data/merged_testing_data.csv"
tested_df.to_csv(output_file, index=False)
print(f"Merged testing dataset saved to {output_file}")

Loading data: 100%|██████████| 3/3 [00:00<00:00, 64.61it/s]


Merged training dataset saved to ../data/merged_training_data.csv


Loading data: 100%|██████████| 2/2 [00:00<00:00, 693.45it/s]

Merged testing dataset saved to ../data/merged_testing_data.csv


In [56]:
# Features to predict
output_vars = ["Total Alkalinity", "Electrical Conductance", "Dissolved Reactive Phosphorus"]

In [57]:
# Check values of data_quality
print("Unique data_quality values:")
print(trained_df["data_quality"].unique())


Unique data_quality values:
['good' 'fair']


In [58]:
import numpy as np
# Create a method to preprocess the data for modeling
def preprocess_data(df):
    # One hot encode data_quality
    df = pd.get_dummies(df, columns=["data_quality"], prefix="data_quality")

    # Split date into day, month, year
    df["Sample Date"] = pd.to_datetime(df["Sample Date"], dayfirst=True, errors='coerce')
    df["day"] = df["Sample Date"].dt.day
    df["month"] = df["Sample Date"].dt.month
    df["year"] = df["Sample Date"].dt.year
    df.drop(columns=["Sample Date", "coastal"], inplace=True)

    return df

# Preprocess the training data
trained_df = preprocess_data(trained_df)
# Preprocess the testing data
tested_df = preprocess_data(tested_df)
# After get_dummies on both, align columns
tested_df = tested_df.reindex(columns=trained_df.columns, fill_value=0)

# Cyclical month encoding
trained_df['month_sin'] = np.sin(2 * np.pi * trained_df['month'] / 12)
trained_df['month_cos'] = np.cos(2 * np.pi * trained_df['month'] / 12)
tested_df['month_sin'] = np.sin(2 * np.pi * tested_df['month'] / 12)
tested_df['month_cos'] = np.cos(2 * np.pi * tested_df['month'] / 12)

# Spatial clusters — fit on train only, apply to both
from sklearn.cluster import KMeans
kmeans = KMeans(n_clusters=50, random_state=42)
trained_df['spatial_cluster'] = kmeans.fit_predict(trained_df[['Latitude', 'Longitude']])
tested_df['spatial_cluster'] = kmeans.predict(tested_df[['Latitude', 'Longitude']])

# Impute remaining NaNs with median per column
from sklearn.impute import SimpleImputer
imputer = SimpleImputer(strategy='median')

# Impute columns except Latitude, Longitude, day, month, year
features = trained_df.drop(columns=output_vars).columns
impute_features = features.difference(["Latitude", "Longitude", "day", "month", "year", "spatial_cluster"])
imputer.fit(trained_df[impute_features])
trained_df[impute_features] = imputer.transform(trained_df[impute_features])
tested_df[impute_features] = imputer.transform(tested_df[impute_features])

# After imputation, BEFORE scaling, add:
for df in [trained_df, tested_df]:
    df['turbidity_x_rain'] = df['TurbidityIndex'] * df['ppt']
    df['runoff_x_soil']    = df['q'] * df['soil']
    df['ndvi_x_rain']      = df['NDVI'] * df['ppt']

# Run standard scaling on the features
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
# scale all features except Latitude, Longitude, day, month, year
# Explicitly exclude targets from scaling
features_to_scale = [c for c in trained_df.columns 
                     if c not in output_vars + ["Latitude", "Longitude", "day", "month", "year", "coastal", "spatial_cluster"]]

trained_df[features_to_scale] = scaler.fit_transform(trained_df[features_to_scale])
tested_df[features_to_scale] = scaler.transform(tested_df[features_to_scale])

/var/folders/yk/c0cfqj715njfdnphhzw8m5p00000gn/T/ipykernel_2315/2227482167.py:8: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  df["Sample Date"] = pd.to_datetime(df["Sample Date"], dayfirst=True, errors='coerce')
/var/folders/yk/c0cfqj715njf

In [59]:
print("Any targets scaled?", any(t in features_to_scale for t in output_vars))
print("Spatial cluster scaled?", 'spatial_cluster' in features_to_scale)
print("Train NaNs:", trained_df.isnull().sum().sum())
print("Test NaNs:", tested_df.isnull().sum().sum())

Any targets scaled? False
Spatial cluster scaled? False
Train NaNs: 0
Test NaNs: 0


In [60]:
tested_df.columns

Index(['Latitude', 'Longitude', 'pet', 'aet', 'def', 'ppt', 'q', 'soil',
       'pdsi', 'tmax', 'tmin', 'vap', 'swe', 'temp_range', 'aridity_index',
       'evap_fraction', 'water_balance', 'Total Alkalinity',
       'Electrical Conductance', 'Dissolved Reactive Phosphorus', 'blue',
       'green', 'red', 'nir', 'swir16', 'swir22', 'NDVI', 'EVI', 'SAVI',
       'ARVI', 'GNDVI', 'RDVI', 'NDWI', 'MNDWI', 'AWEInsh', 'AWEIsh', 'BSI',
       'NDBI', 'UI', 'NBR', 'ClayMinerals', 'TurbidityIndex',
       'ChlorophyllIndex', 'NIR_Red_Ratio', 'NDMI', 'cloud_cover',
       'data_quality_fair', 'data_quality_good', 'day', 'month', 'year',
       'month_sin', 'month_cos', 'spatial_cluster', 'turbidity_x_rain',
       'runoff_x_soil', 'ndvi_x_rain'],
      dtype='object')

In [65]:
# Try without spatial_cluster
trained_df.drop(columns=['spatial_cluster'], inplace=True)
tested_df.drop(columns=['spatial_cluster'], inplace=True)

In [ ]:
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split, GridSearchCV
import numpy as np

sqrt_targets = ['Dissolved Reactive Phosphorus']
targets = output_vars

transforms = {
    'Total Alkalinity': 'none',
    'Electrical Conductance': 'none',
    'Dissolved Reactive Phosphorus': 'sqrt'
}

predictions = {}
val_r2s = {}

southern_mask = trained_df['Latitude'] < -30
southern_train = trained_df[southern_mask]
print(f"Southern training samples: {len(southern_train)}")

# Use only southern samples for training
X = southern_train.drop(columns=targets)
y_all = southern_train[targets].copy()

for target in sqrt_targets:
    y_all[target] = np.sqrt(y_all[target])

X_train, X_val = train_test_split(X, test_size=0.2, random_state=42)
X_test = tested_df.drop(columns=targets, errors='ignore')

for target in targets:
    print(f"\n{'='*50}")
    print(f"Training models for: {target}")

    y_train = y_all.loc[X_train.index, target]
    y_val   = y_all.loc[X_val.index,   target]
    y_full  = y_all[target]

    # Tune LGBM on train split
    grid_search = GridSearchCV(
        LGBMRegressor(random_state=42, verbose=-1),
        param_grid={
            'n_estimators':     [200, 500],
            'num_leaves':       [31, 63, 127],
            'max_depth':        [-1, 7],
            'learning_rate':    [0.05, 0.1],
            'subsample':        [0.8],
            'colsample_bytree': [0.8],
            'min_child_samples':[10, 20],
        },
        cv=3, scoring='r2', n_jobs=-1, verbose=0
    )
    grid_search.fit(X_train, y_train)
    best_params = grid_search.best_params_
    print(f"  Best params: {best_params}")

    # Val R² using LGBM with best params
    lgbm_val = LGBMRegressor(**best_params, random_state=42, verbose=-1)
    lgbm_val.fit(X_train, y_train)
    y_pred_t = lgbm_val.predict(X_val)
    if transforms[target] == 'sqrt':
        y_pred     = np.square(y_pred_t)
        y_val_orig = np.square(y_val)
    else:
        y_pred     = y_pred_t
        y_val_orig = y_val
    r2 = r2_score(y_val_orig, y_pred)
    print(f"  Val R²: {r2:.4f}")
    val_r2s[target] = r2

    # Retrain all three on FULL data
    lgbm = LGBMRegressor(**best_params, random_state=42, verbose=-1)
    lgbm.fit(X, y_full)

    max_depth = best_params.get('max_depth', 6)
    if max_depth == -1:
        max_depth = 6
    xgb = XGBRegressor(
        n_estimators=best_params.get('n_estimators', 500),
        max_depth=max_depth,
        learning_rate=best_params.get('learning_rate', 0.05),
        subsample=0.8, colsample_bytree=0.8,
        objective='reg:squarederror', random_state=42
    )
    xgb.fit(X, y_full)

    rf = RandomForestRegressor(n_estimators=300, max_depth=10, random_state=42, n_jobs=-1)
    rf.fit(X, y_full)

    # Average ensemble predictions
    avg_preds = (lgbm.predict(X_test) + xgb.predict(X_test) + rf.predict(X_test)) / 3
    predictions[target] = np.square(avg_preds) if transforms[target] == 'sqrt' else avg_preds

print(f"\nMean Val R²: {sum(val_r2s.values()) / len(val_r2s):.4f}")
for t, r2 in val_r2s.items():
    print(f"  {t}: {r2:.4f}")

Southern training samples: 2369

Training models for: Total Alkalinity


/var/folders/yk/c0cfqj715njfdnphhzw8m5p00000gn/T/ipykernel_2315/681346477.py:32: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  y_all[target] = np.sqrt(y_all[target])
/Users/shreyasriram/personal_projects/ey/.venv/lib/python3.14/site-packages

  Best params: {'colsample_bytree': 0.8, 'learning_rate': 0.05, 'max_depth': 7, 'min_child_samples': 10, 'n_estimators': 200, 'num_leaves': 31, 'subsample': 0.8}
  Val R²: 0.8015

Training models for: Electrical Conductance


/Users/shreyasriram/personal_projects/ey/.venv/lib/python3.14/site-packages/numpy/ma/core.py:2885: RuntimeWarning: invalid value encountered in cast
  _data = np.array(data, dtype=dtype, copy=copy,


  Best params: {'colsample_bytree': 0.8, 'learning_rate': 0.05, 'max_depth': -1, 'min_child_samples': 20, 'n_estimators': 200, 'num_leaves': 63, 'subsample': 0.8}
  Val R²: 0.7226

Training models for: Dissolved Reactive Phosphorus


/Users/shreyasriram/personal_projects/ey/.venv/lib/python3.14/site-packages/numpy/ma/core.py:2885: RuntimeWarning: invalid value encountered in cast
  _data = np.array(data, dtype=dtype, copy=copy,


  Best params: {'colsample_bytree': 0.8, 'learning_rate': 0.05, 'max_depth': 7, 'min_child_samples': 10, 'n_estimators': 200, 'num_leaves': 63, 'subsample': 0.8}
  Val R²: 0.3605

Mean Val R²: 0.6282
  Total Alkalinity: 0.8015
  Electrical Conductance: 0.7226
  Dissolved Reactive Phosphorus: 0.3605


In [73]:
# Load the original template to get the correct row order
template = pd.read_csv('../data/output/submission_template.csv')

test_predictions = pd.DataFrame(predictions)
X_test = tested_df.copy()

test_predictions['Latitude'] = X_test['Latitude'].values
test_predictions['Longitude'] = X_test['Longitude'].values
test_predictions['Sample Date'] = (
    X_test['day'].astype(int).astype(str).str.zfill(2) + '-' +
    X_test['month'].astype(int).astype(str).str.zfill(2) + '-' +
    X_test['year'].astype(int).astype(str)
)

test_predictions = test_predictions[["Latitude", "Longitude", "Sample Date"] + targets]

# Merge with template to enforce correct row order
template['Sample Date'] = pd.to_datetime(template['Sample Date'], dayfirst=True, errors='coerce').dt.strftime('%d-%m-%Y')
test_predictions['Sample Date'] = pd.to_datetime(test_predictions['Sample Date'], dayfirst=True, errors='coerce').dt.strftime('%d-%m-%Y')

ordered = template[['Latitude', 'Longitude', 'Sample Date']].merge(
    test_predictions,
    on=['Latitude', 'Longitude', 'Sample Date'],
    how='left'
)

# Verify no rows lost
print(f"Template rows: {len(template)}, Output rows: {len(ordered)}")
print(f"Missing predictions: {ordered[targets].isnull().sum().sum()}")
print(f"Sample dates: {ordered['Sample Date'].head()}")
print(f"Prediction ranges:")
for target in targets:
    print(f"  {target}: {ordered[target].min():.2f} - {ordered[target].max():.2f}")

ordered.to_csv('../data/modeling/test_predictions_final.csv', index=False)
print("Saved!")

Template rows: 200, Output rows: 200
Missing predictions: 0
Sample dates: 0    01-09-2014
1    16-09-2015
2    07-05-2015
3    07-02-2012
4    01-10-2014
Name: Sample Date, dtype: object
Prediction ranges:
  Total Alkalinity: 20.98 - 200.11
  Electrical Conductance: 160.56 - 838.39
  Dissolved Reactive Phosphorus: 10.06 - 71.08
Saved!


/var/folders/yk/c0cfqj715njfdnphhzw8m5p00000gn/T/ipykernel_2315/2428004605.py:18: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  template['Sample Date'] = pd.to_datetime(template['Sample Date'], dayfirst=True, errors='coerce').dt.strftime('%d

In [63]:
# Compare feature distributions between train and test
X_test = tested_df.drop(columns=targets, errors='ignore')
X_train_full = trained_df.drop(columns=targets)

print("=== Feature distribution comparison ===")
for col in ['Latitude', 'Longitude', 'month', 'year', 'pet', 'ppt', 'NDVI']:
    print(f"\n{col}:")
    print(f"  Train: mean={X_train_full[col].mean():.3f}, std={X_train_full[col].std():.3f}, min={X_train_full[col].min():.3f}, max={X_train_full[col].max():.3f}")
    print(f"  Test:  mean={X_test[col].mean():.3f}, std={X_test[col].std():.3f}, min={X_test[col].min():.3f}, max={X_test[col].max():.3f}")

=== Feature distribution comparison ===

Latitude:
  Train: mean=-28.475, std=2.760, min=-34.406, max=-22.226
  Test:  mean=-32.817, std=0.652, min=-34.096, max=-31.903

Longitude:
  Train: mean=26.868, std=3.535, min=17.730, max=32.325
  Test:  mean=26.583, std=1.322, min=24.196, max=28.582

month:
  Train: mean=6.445, std=3.333, min=1.000, max=12.000
  Test:  mean=6.395, std=3.304, min=1.000, max=12.000

year:
  Train: mean=2013.105, std=1.384, min=2011.000, max=2015.000
  Test:  mean=2013.295, std=1.344, min=2011.000, max=2015.000

pet:
  Train: mean=-0.000, std=1.000, min=-4.156, max=3.245
  Test:  mean=-0.505, std=0.860, min=-3.891, max=1.460

ppt:
  Train: mean=0.000, std=1.000, min=-1.267, max=5.476
  Test:  mean=-0.363, std=0.695, min=-1.180, max=1.961

NDVI:
  Train: mean=-0.000, std=1.000, min=-6.097, max=3.832
  Test:  mean=0.415, std=1.064, min=-2.871, max=2.747


In [64]:
print(ordered.shape)
print(ordered.isnull().sum())
print(ordered.head(10))

(200, 6)
Latitude                         0
Longitude                        0
Sample Date                      0
Total Alkalinity                 0
Electrical Conductance           0
Dissolved Reactive Phosphorus    0
dtype: int64
    Latitude  Longitude Sample Date  Total Alkalinity  Electrical Conductance  \
0 -32.043333  27.822778  01-09-2014         81.131260              313.923532   
1 -33.329167  26.077500  16-09-2015        145.484604              713.158694   
2 -32.991639  27.640028  07-05-2015         74.835228              298.373921   
3 -34.096389  24.439167  07-02-2012         36.523687              423.230373   
4 -32.000556  28.581667  01-10-2014         78.086506              245.146352   
5 -32.086390  25.575560  19-07-2013        141.505292              663.063378   
6 -32.000556  28.581667  03-09-2014         78.874151              249.577590   
7 -32.991639  27.640028  02-10-2014         74.338364              328.037765   
8 -32.000556  28.581667  06-08-2014    

In [68]:
# See how many training samples are in the test's geographic range
southern_train = trained_df[trained_df['Latitude'] < -31.9]
print(f"Training samples below lat -31.9: {len(southern_train)} out of {len(trained_df)}")
print(southern_train[['Latitude', 'Longitude']].describe())

Training samples below lat -31.9: 1230 out of 9319
          Latitude    Longitude
count  1230.000000  1230.000000
mean    -33.688342    20.294655
std       0.529950     1.315870
min     -34.405833    18.336670
25%     -34.039722    19.074722
50%     -33.897778    20.003333
75%     -33.380560    21.624167
max     -32.502778    22.548333


In [70]:
# check if any values in the test set are outside the range of the training set for key features
for feature in features:
    train_min = X_train_full[feature].min()
    train_max = X_train_full[feature].max()
    test_min = X_test[feature].min()
    test_max = X_test[feature].max()
    print(f"{feature}: Train range=({train_min:.3f}, {train_max:.3f}), Test range=({test_min:.3f}, {test_max:.3f})")

Latitude: Train range=(-34.406, -22.226), Test range=(-34.096, -31.903)
Longitude: Train range=(17.730, 32.325), Test range=(24.196, 28.582)
pet: Train range=(-4.156, 3.245), Test range=(-3.891, 1.460)
aet: Train range=(-1.442, 2.377), Test range=(-1.343, 1.461)
def: Train range=(-1.652, 2.325), Test range=(-1.652, 1.566)
ppt: Train range=(-1.267, 5.476), Test range=(-1.180, 1.961)
q: Train range=(-0.543, 15.760), Test range=(-0.517, 0.595)
soil: Train range=(-0.508, 8.984), Test range=(-0.502, 3.132)
pdsi: Train range=(-2.250, 3.656), Test range=(-0.670, 2.195)
tmax: Train range=(-5.299, 2.844), Test range=(-3.567, 1.518)
tmin: Train range=(-7.461, 2.525), Test range=(-4.418, 1.106)
vap: Train range=(-3.892, 2.859), Test range=(-2.821, 1.567)
swe: Train range=(0.000, 0.000), Test range=(0.000, 0.000)
temp_range: Train range=(-3.064, 3.142), Test range=(-2.510, 1.699)
aridity_index: Train range=(-1.110, 6.326), Test range=(-1.054, 2.459)
evap_fraction: Train range=(-1.297, 1.884), Test

KeyError: 'spatial_cluster'

In [72]:
# range of values of output variables in the train set
print("\nOutput variable ranges in training set:")
for target in targets:
    print(f"{target}: min={y_all[target].min():.3f}, max={y_all[target].max():.3f}")
    if target in sqrt_targets:
        print(f"  (sqrt transformed)")
        print(f"  (original) min={np.square(y_all[target].min()):.3f}, max={np.square(y_all[target].max()):.3f}")



Output variable ranges in training set:
Total Alkalinity: min=4.800, max=361.492
Electrical Conductance: min=15.120, max=1505.000
Dissolved Reactive Phosphorus: min=3.162, max=13.766
  (sqrt transformed)
  (original) min=10.000, max=189.500
